In [ ]:
import json
import os.path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.font_manager import FontProperties
from scipy.stats import shapiro, wilcoxon

In [ ]:
model = "Qwen/Qwen2.5-Coder-32B"
short_model_name = model.split("/")[-1]
model_results_dir = f"../model_results/{short_model_name}"
model_pairwise_results_dir = os.path.join(model_results_dir, "pairwise_analysis")
model_pairwise_comparisons_dir = os.path.join(model_pairwise_results_dir, "pairwise_comparisons")
output_dir = "RQ1"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
plt.rcParams.update({'font.size': 11, 'font.family': 'sans-serif'})

In [ ]:
def load_snippet(json_path, key):
    with open(json_path, "r") as f:
        data = json.load(f)
    entry = data[key]
    return entry["code"], entry["tokens"], entry["results"], entry["summary"]


def get_perplexities(results):
    return [r["perplexity"] for r in results]


import matplotlib.patches as patches


def plot_code_block_with_aoi(ax, code, fontsize, tokens, highlight_ranges=None, snippet_type="clean", padding=0.01):
    ax.axis("off")

    font_prop = FontProperties(family="monospace", size=fontsize)
    line_spacing = 1.5

    code_lines = code.splitlines()

    # Inverted transform for converting display to axes coords
    inv = ax.transAxes.inverted()

    y_cursor = 1  # start at top (axes coords)

    token_idx = 0
    token_positions = []

    # Calculate single char height in display coords
    sample_char = "A"
    tp = ax.text(0, 0, sample_char, fontproperties=font_prop)
    renderer = ax.figure.canvas.get_renderer()
    bbox = tp.get_window_extent(renderer=renderer)
    char_height_disp = bbox.height
    tp.remove()

    # Convert char height from display to axes coords for line height
    p0 = inv.transform((0, 0))
    p1 = inv.transform((0, char_height_disp))
    line_height = (p1[1] - p0[1]) * line_spacing

    for line in code_lines:
        x_cursor = padding

        # Collect tokens that belong to this line
        line_length = len(line)
        length_sum = 0
        line_tokens = []
        while token_idx < len(tokens) and length_sum < line_length:
            t = tokens[token_idx].replace("\n", "")
            length_sum += len(t)
            line_tokens.append(tokens[token_idx])
            token_idx += 1

        for t in line_tokens:
            t_stripped = t.replace("\n", "")
            if not t_stripped:
                continue

            # Draw token text individually at (x_cursor, y_cursor) in axes coords
            txt = ax.text(x_cursor, y_cursor, t_stripped,
                          fontproperties=font_prop,
                          va='top', ha='left',
                          transform=ax.transAxes)

            # Get bounding box in display coords
            bbox = txt.get_window_extent(renderer=ax.figure.canvas.get_renderer())

            # Convert bbox corners from display to axes coords
            fig = ax.figure
            fig_height_disp = fig.bbox.height

            x0_disp = bbox.x0
            y0_disp = bbox.y0
            x1_disp = bbox.x1
            y1_disp = bbox.y1

            # Invert y because display y coords go from top to bottom
            y0_disp_inverted = fig_height_disp - y0_disp
            y1_disp_inverted = fig_height_disp - y1_disp

            x0_ax, y0_ax = inv.transform((x0_disp, y0_disp_inverted))
            x1_ax, y1_ax = inv.transform((x1_disp, y1_disp_inverted))

            token_positions.append([x0_ax, y0_ax, x1_ax, y1_ax])

            x_cursor = x1_ax + padding  # advance cursor with padding in axes coords

        y_cursor -= line_height

    # Set colors for AOI boxes
    if snippet_type == "clean":
        edge_color = "#1F77B4"  # blue
        fill_color = "#ADD8E6"  # light blue
    else:
        edge_color = "#D62728"  # red
        fill_color = "#F4C6C5"  # light red

    # Draw AOI boxes if any
    if highlight_ranges:
        for start, end in highlight_ranges:
            if start < 0 or end >= len(token_positions) or start > end:
                continue

            x0 = min(token_positions[i][0] for i in range(start, end + 1))
            x1 = max(token_positions[i][2] for i in range(start, end + 1))
            y0 = min(token_positions[i][1] for i in range(start, end + 1))
            y1 = max(token_positions[i][3] for i in range(start, end + 1))

            # Add padding and clamp to axes limits
            x0_p = max(0, x0 - padding)
            y0_p = max(0, y0 - padding) + 0.22
            x1_p = min(1, x1 + padding)
            y1_p = min(1, y1 + padding)

            width = x1_p - x0_p
            height = line_height

            rect = patches.Rectangle(
                (x0_p, y0_p), width, height,
                linewidth=1.5,
                edgecolor=edge_color,
                facecolor=fill_color,
                alpha=0.4,
                transform=ax.transAxes,
                joinstyle='round',
                capstyle='round'
            )
            ax.add_patch(rect)

    # Adjust axes limits tightly
    ax.set_xlim(0, 1)
    ax.set_ylim(y_cursor, 1)


def plot_token_bars(ax, tokens, perplexities, fontsize, highlight_ranges=None, snippet_type="clean", label=False,
                    ymax=None):
    """
    Plot token bars with log-scaled perplexity values.
    highlight_ranges: list of (start_idx, end_idx) tuples to highlight tokens in these ranges.
    """
    font_prop = FontProperties(family="monospace", size=fontsize)

    log_pplx = [np.log1p(p) for p in perplexities]
    max_log = max(log_pplx) if ymax is None else ymax
    x = np.arange(len(tokens))

    # Define colors based on snippet type
    if snippet_type == "clean":
        base_fill = "#D3D3D3"  # light gray for non-highlight
        base_edge = "none"
        highlight_fill = "#1F77B4"  # blue
        highlight_fill = highlight_fill + "CC"  # add transparency with alpha hex (CC ~ 80% opacity)
        highlight_edge = "none"  # darker blue

    else:  # confusing snippet
        base_fill = "#D3D3D3"  # light gray for non-highlight
        base_edge = "none"
        highlight_fill = "#D62728"  # red
        highlight_fill = highlight_fill + "CC"  # add transparency with alpha hex (CC ~ 80% opacity)
        highlight_edge = "none"  # dark red

    # Prepare colors per bar
    colors = [base_fill] * len(tokens)
    edges = [base_edge] * len(tokens)

    if highlight_ranges:
        for start, end in highlight_ranges:
            for i in range(start, end + 1):
                if 0 <= i < len(tokens):
                    colors[i] = highlight_fill
                    edges[i] = highlight_edge

    # Draw bars individually to assign different colors
    for i in range(len(tokens)):
        ax.bar(x[i], log_pplx[i], color=colors[i], edgecolor=edges[i], linewidth=0.7, width=0.8)

    ax.set_ylim(0, max_log * 1.1)
    ax.set_xlim(-0.5, len(tokens) - 0.5)

    # Subtle horizontal grid lines
    ax.yaxis.grid(True, linestyle='--', alpha=0.3, linewidth=0.6)

    # Token labels rotated 90 degrees, below bars
    # for i, token in enumerate(tokens):
    #    ax.text(x[i], -max_log * 0.07, token.replace("\n", "\\n"),
    #            rotation=90, fontsize=fontsize, ha="center",
    #            va="top", family=monospace_font)

    for i, token in enumerate(tokens):
        if any(start <= i <= end for start, end in highlight_ranges):
            ax.text(
                x[i], -max_log * 0.07, token.replace("\n", "\\n"),
                rotation=0, ha="center",
                va="top", fontproperties=font_prop
            )

    # Remove spines except bottom
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(base_edge)
    ax.spines["bottom"].set_linewidth(0.8)

    # remove x-axis ticks and labels
    ax.set_xticks(x)
    ax.set_xticklabels([])
    ax.tick_params(axis='x', length=0)
    ax.xaxis.set_label_position('top')
    ax.set_xlabel("Tokens", fontsize=fontsize, labelpad=10)

    # Remove y-axis ticks and labels
    ax.set_yticks([])
    if label:
        ax.set_ylabel("LLM\nPerplexity", fontsize=fontsize)
    ax.tick_params(axis='y', length=0)

    # Slight padding below bars for labels
    ax.margins(y=0.2)

In [ ]:
# JSON file containing perplexity results
model_results_dir = f"../model_results/{short_model_name}"
json_file = f"{model_results_dir}/perplexity_results.json"

# keys for the JSON file
clean_key = "7-clean-v1"
confusing_key = "7-obf-v1"

# font properties for the plot
monospace_font = "Courier New"
pixels_per_token = 0.28
base_fontsize = 13
bar_fontsize = 9
bar_color = "#D62728"  # red tone
bar_linewidth = 0.4

# highlight token index ranges here (inclusive, zero-based)
highlight_ranges_clean = [(10, 11)]
highlight_ranges_conf = [(10, 12)]

# ---------- Load ----------
code_clean, tokens_clean, results_clean, summary_clean = load_snippet(json_file, clean_key)
code_conf, tokens_conf, results_conf, summary_conf = load_snippet(json_file, confusing_key)

pplx_clean = get_perplexities(results_clean)
pplx_conf = get_perplexities(results_conf)

log_clean = [np.log1p(p) for p in pplx_clean]
log_conf = [np.log1p(p) for p in pplx_conf]
global_max_log = max(max(log_clean), max(log_conf))

# ---------- Dynamic Figure Size ----------
num_tokens = max(len(tokens_clean), len(tokens_conf))
width = pixels_per_token * num_tokens * 2.0
width = min(width, 8.5)
height = 4.5

fig = plt.figure(figsize=(width, height))
spec = gridspec.GridSpec(ncols=2, nrows=3, figure=fig,
                         height_ratios=[1, 1, 1], hspace=1, wspace=0.1)

# Subplots
ax_code_clean = fig.add_subplot(spec[0, 0])
ax_code_conf = fig.add_subplot(spec[0, 1])
ax_pplx_clean = fig.add_subplot(spec[1, 0])
ax_pplx_conf = fig.add_subplot(spec[1, 1])

# ---------- Plot ----------
plot_code_block_with_aoi(ax_code_clean, code_clean, base_fontsize + 2, tokens_clean, highlight_ranges_clean, "clean")
plot_code_block_with_aoi(ax_code_conf, code_conf, base_fontsize + 2, tokens_conf, highlight_ranges_conf, "confusing")
plot_token_bars(ax_pplx_clean, tokens_clean, pplx_clean, bar_fontsize, highlight_ranges_clean, "clean", label=True,
                ymax=global_max_log)
plot_token_bars(ax_pplx_conf, tokens_conf, pplx_conf, bar_fontsize, highlight_ranges_conf, "confusing", label=False,
                ymax=global_max_log)

fig.align_ylabels()
# print clean and confusing summaries
print("Clean Snippet Summary:")
print(summary_clean)
print("\nConfusing Snippet Summary:")
print(summary_conf)
# Add column labels on top
fig.text(0.3, 0.95, "Clean", fontsize=base_fontsize, ha="center", va="bottom")
fig.text(0.7, 0.95, "Confusing", fontsize=base_fontsize, ha="center", va="bottom")

# Save the figure
plt.savefig(os.path.join(output_dir, "rq1_perplexity_diff_visualization.pdf"), dpi=300, bbox_inches='tight',
            pad_inches=0)

In [ ]:
import numpy as np
from scipy.stats import wilcoxon, rankdata

def wilcoxon_signed_rank_effect_size(x, y):
    """
    Perform Wilcoxon signed-rank test on paired samples x and y,
    and compute effect size r with correct sign.

    Returns:
        w: Wilcoxon test statistic (sum of ranks of the less frequent sign)
        p: p-value
        r: effect size (z / sqrt(n)) with sign indicating direction
    """
    diffs = np.array(x) - np.array(y)

    # Remove zero differences because Wilcoxon ignores them
    nonzero_diffs = diffs[diffs != 0]
    n_nonzero = len(nonzero_diffs)

    if n_nonzero == 0:
        raise ValueError("All differences are zero.")

    # Perform Wilcoxon test
    w, p = wilcoxon(nonzero_diffs)

    # Calculate ranks of absolute differences
    ranks = rankdata(np.abs(nonzero_diffs))

    # Sum ranks for positive and negative differences
    pos_rank_sum = ranks[nonzero_diffs > 0].sum()
    neg_rank_sum = ranks[nonzero_diffs < 0].sum()

    # Mean and std dev of W under null hypothesis
    mean_W = n_nonzero * (n_nonzero + 1) / 4
    std_W = np.sqrt(n_nonzero * (n_nonzero + 1) * (2 * n_nonzero + 1) / 24)

    # Determine the sign of the effect size based on which side has smaller rank sum
    # Wilcoxon statistic W is min(pos_rank_sum, neg_rank_sum)
    sign = 1 if pos_rank_sum < neg_rank_sum else -1

    # Compute z-score (standardized test statistic)
    z = (w - mean_W) / std_W
    z *= sign  # apply sign to indicate direction

    # Compute effect size r
    r = z / np.sqrt(n_nonzero)

    return w, p, r


In [ ]:
def combined_paired_ttest_perplexity_metrics(results_path):
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import pandas as pd
    import numpy as np
    from scipy import stats

    df = pd.read_csv(results_path)
    clean_df = df[~df['File'].str.contains('obf')]
    obf_df = df[df['File'].str.contains('obf')]

    metrics_info = [
        ("Overall Perplexity", "Snippet-Level PP"),
        ("Max Perplexity", "Max Snippet-Level PP"),
        ("Overall AOI Perplexity", "AOI-Level PP"),
        ("Max AOI Perplexity", "Max AOI-Level PP")
    ]

    group_spacing = 0.65
    spacing = 0.15
    offsets = {label: i * group_spacing for i, (_, label) in enumerate(metrics_info)}
    paired_data = {label: {'clean': [], 'obf': []} for _, label in metrics_info}

    for _, clean_row in clean_df.iterrows():
        parts = clean_row['File'].split('-')
        if len(parts) >= 3:
            sid, ver = parts[0], parts[-1]
            obf_name = f"{sid}-obf-{ver}"
            match = obf_df[obf_df['File'] == obf_name]
            if not match.empty:
                obf_row = match.iloc[0]
                for col, label in metrics_info:
                    paired_data[label]['clean'].append(clean_row[col])
                    paired_data[label]['obf'].append(obf_row[col])

    short_labels = ["Avg. Perplexity", "Max. Perplexity", "Avg. Perplexity", "Max. Perplexity"]
    indiv_positions = [offsets[label] for _, label in metrics_info]
    group_positions = [
        (indiv_positions[0] + indiv_positions[1]) / 2,
        (indiv_positions[2] + indiv_positions[3]) / 2
    ]

    fig, ax = plt.subplots(figsize=(5, 2), dpi=300)

    color_clean = '#1F77B4'
    color_obf = '#D62728'
    edge_color = 'black'

    all_clean, all_obf = [], []
    max_y = 0

    midpoint = (group_positions[0] + group_positions[1]) / 2
    ax.axvline(midpoint, color='gray', linestyle='--', linewidth=0.8, zorder=0)

    for _, label in metrics_info:
        x = offsets[label]
        clean_vals = np.array(paired_data[label]['clean'])
        obf_vals = np.array(paired_data[label]['obf'])
        if len(clean_vals) == 0:
            continue

        all_obf.append((x - spacing, obf_vals))
        all_clean.append((x + spacing, clean_vals))
        max_y = max(max_y, clean_vals.max(), obf_vals.max())

        for yc, yo in zip(clean_vals, obf_vals):
            ax.plot([x - spacing, x + spacing], [yo, yc], color='gray', alpha=0.15, linewidth=0.7, zorder=1)
        ax.scatter(np.full_like(obf_vals, x - spacing), obf_vals, color='gray', alpha=0.40, s=9, zorder=2)
        ax.scatter(np.full_like(clean_vals, x + spacing), clean_vals, color='gray', alpha=0.40, s=9, zorder=2)

    box_data = []
    box_positions = []
    box_colors = []

    box_width = 0.1

    for i in range(len(metrics_info)):
        pos_clean, vals_clean = all_clean[i]
        pos_obf, vals_obf = all_obf[i]

        box_data.append(vals_obf)
        box_positions.append(pos_obf)
        box_colors.append(color_obf)

        box_data.append(vals_clean)
        box_positions.append(pos_clean)
        box_colors.append(color_clean)

    bplots = ax.boxplot(
        box_data,
        positions=box_positions,
        widths=box_width,
        patch_artist=True,
        boxprops=dict(color=edge_color, linewidth=0.8),
        medianprops=dict(color=edge_color, linewidth=1),
        whiskerprops=dict(color=edge_color, linewidth=0.8),
        capprops=dict(color=edge_color, linewidth=0.8),
        flierprops=dict(marker='', linestyle='none'),
        zorder=3
    )
    for patch, fc in zip(bplots['boxes'], box_colors):
        rgba = mcolors.to_rgba(fc, alpha=0.60)
        patch.set_facecolor(rgba)

    ann_y = max_y * 1.25
    for i, (_, label) in enumerate(metrics_info):
        c = np.array(paired_data[label]['clean'])
        o = np.array(paired_data[label]['obf'])
        if len(c) == 0 or len(o) == 0:
            continue

        diffs = o - c
        # Check for normality of the differences
        _, normality_p = shapiro(diffs)
        print(f"Normality test for {label}: p = {normality_p:.3f}")

        if normality_p > 0.05:
            print(f"{label} differences are normally distributed, using paired t-test.")
            # Paired t-test
            t, p = stats.ttest_rel(o, c)
            d = np.mean(diffs) / np.std(diffs, ddof=1)
            test_stat = f"$t = {t:.2f}$"
            eff_size = f"$d = {d:.2f}$"
        else:
            print(f"{label} differences are not normally distributed, using Wilcoxon signed-rank test.")
            # Wilcoxon signed-rank test
            w, p, r = wilcoxon_signed_rank_effect_size(o, c)
            print(f"{label} Wilcoxon test statistic: {w}, p-value: {p:.3f}, effect size r: {r:.2f}")
            test_stat = f"$W = {w:.0f}$"
            eff_size = f"$r = {r:.3f}$"

        # Format p-value
        if p < 0.001:
            fmt_p = r"$\mathbf{p < 0.001}$"
        elif p < 0.05:
            fmt_p = fr"$\mathbf{{p = {p:.3f}}}$"
        else:
            fmt_p = fr"$p = {p:.2f}$"

        txt = f"{test_stat}\n{fmt_p}\n{eff_size}"

        center_x = (box_positions[2 * i] + box_positions[2 * i + 1]) / 2
        ax.text(center_x, ann_y, txt,
                ha='center', va='bottom', fontsize=8,
                bbox=dict(facecolor='gray', alpha=0.15, edgecolor='none', boxstyle='round,pad=0.3'),
                zorder=4)

    ax.set_xticks(indiv_positions)
    ax.set_xticklabels(short_labels, ha='center', fontsize=9)
    ax.set_ylabel("Perplexity", labelpad=6)
    ax.set_yscale("log")
    ax.tick_params(axis='y', labelsize=9)
    ax.tick_params(axis='x', length=0)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.7)
    ax.spines['left'].set_linewidth(0.7)
    ax.grid(False)

    # Add group labels below x-axis using axes-relative coordinates
    ax.text(0.25, -0.18, "Snippet-Level", ha='center', va='top',
            transform=ax.transAxes, fontweight='bold')
    ax.text(0.75, -0.18, "AOI-Level", ha='center', va='top',
            transform=ax.transAxes, fontweight='bold')

    plt.subplots_adjust(left=0.08, right=0.97, top=0.88, bottom=0.18)

    # Tight x-axis limits
    pad = 0.2
    xmin = min(box_positions) - pad
    xmax = max(box_positions) + pad
    ax.set_xlim(xmin, xmax)

    plt.savefig(f"{output_dir}/combined_paired_ttest_perplexity_metrics.pdf", bbox_inches='tight', dpi=300,
                pad_inches=0.05)
    plt.show()


In [ ]:
combined_paired_ttest_perplexity_metrics(results_path=f"{model_pairwise_comparisons_dir}/perplexity_stats.csv")